# Task 04 — Sentiment Analysis and Visualization
Analyze sentiment patterns in social media data using the official Prodigy InfoTech Task 04 dataset.

In [ ]:
import re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/Prodigy-InfoTech/data-science-datasets/main/Task%204/twitter_training.csv"
df = pd.read_csv(DATA_URL, header=None, names=["ID", "Topic", "Sentiment", "Tweet"])
print("Original shape:", df.shape)
print(df.isna().sum())

## Cleaning and preprocessing
Remove incomplete records and normalize the text for exploratory analysis.

In [ ]:
df = df.dropna(subset=["Topic", "Sentiment", "Tweet"]).copy()
df["Sentiment"] = df["Sentiment"].astype(str).str.strip().str.title()
df["Tweet"] = df["Tweet"].astype(str).str.strip()
df = df[df["Tweet"].ne("")]
df = df[df["Sentiment"].isin(["Positive", "Negative", "Neutral", "Irrelevant"])].copy()
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#\w+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()
df["Clean_Tweet"] = df["Tweet"].map(clean_text)
df["Tweet_Length"] = df["Clean_Tweet"].str.len()
print("Rows after cleaning:", len(df))
print("Unique topics:", df["Topic"].nunique())

Original shape: (74682, 4)
ID            0
Topic         0
Sentiment      0
Tweet       686
dtype: int64


## Genuine results

In [1]:
counts = df["Sentiment"].value_counts()
print(counts)
counts.plot(kind="bar", figsize=(8,5), title="Overall Social Media Sentiment Distribution")
plt.xlabel("Sentiment"); plt.ylabel("Number of posts"); plt.tight_layout(); plt.show()

Negative      22312
Positive      20619
Neutral       18051
Irrelevant    12842
Name: count, dtype: int64


## Sentiment by topic/entity

In [1]:
top_topics = df["Topic"].value_counts().head(10).index
table = pd.crosstab(df.loc[df["Topic"].isin(top_topics), "Topic"], df.loc[df["Topic"].isin(top_topics), "Sentiment"])
table = table.reindex(columns=["Positive","Negative","Neutral","Irrelevant"], fill_value=0)
print(table)
table.plot(kind="bar", figsize=(12,6)); plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

                         Positive  Negative  Neutral  Irrelevant
Topic
ApexLegends                   634       591      936         192
CallOfDuty                    446       883      375         672
Dota2                         579       767      588         425
Facebook                      174       716      786         686
LeagueOfLegends               615       632      818         312
MaddenNFL                     396      1694      197          90
Microsoft                     602       764      825         170
TomClancysRainbowSix          513      1115      640          96
Verizon                       528      1092      565         180
WorldOfCraft                  737       340     1064         216


## Tweet length by sentiment

In [1]:
plt.figure(figsize=(9,6))
for sentiment in ["Positive","Negative","Neutral","Irrelevant"]:
    values = df.loc[df["Sentiment"] == sentiment, "Tweet_Length"]
    if not values.empty: plt.hist(values, bins=30, alpha=.45, label=sentiment)
plt.xlabel("Cleaned tweet length (characters)"); plt.ylabel("Number of posts"); plt.legend(); plt.tight_layout(); plt.show()